# Colab BGE-M3 Reranker API

Colab GPU에는 BGE-M3만 적재하고 로컬 Customer Intelligence가 HTTPS로 `/rerank`를 호출합니다. Colab의 Secrets에 `NGROK_AUTHTOKEN`과 충분히 긴 임의 값인 `RERANKER_API_KEY`를 등록한 뒤 GPU 런타임에서 전체 셀을 실행하세요. 런타임이 종료되면 URL도 만료됩니다.

In [ ]:
%pip install -q FlagEmbedding==1.4.2 fastapi uvicorn pyngrok

In [ ]:
from getpass import getpass
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임을 선택하세요.")

NGROK_AUTHTOKEN = getpass("NGROK_AUTHTOKEN 입력: ").strip()
RERANKER_API_KEY = getpass("RERANKER_API_KEY 입력: ").strip()

if not NGROK_AUTHTOKEN or not RERANKER_API_KEY:
    raise RuntimeError("두 인증값을 모두 입력해야 합니다.")

In [ ]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel('BAAI/bge-m3', devices='cuda', use_fp16=True)
print('loaded: BAAI/bge-m3')

In [ ]:
import secrets
import threading

import uvicorn
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field
from pyngrok import ngrok

class RerankRequest(BaseModel):
    query: str = Field(min_length=1)
    passages: list[str] = Field(min_length=1, max_length=20)

app = FastAPI()

@app.get('/health')
def health():
    return {'status': 'ok', 'model': 'BAAI/bge-m3'}

@app.post('/rerank')
def rerank(payload: RerankRequest, authorization: str | None = Header(default=None)):
    expected = f'Bearer {RERANKER_API_KEY}'
    if authorization is None or not secrets.compare_digest(authorization, expected):
        raise HTTPException(status_code=401, detail='invalid reranker token')
    pairs = [(payload.query, passage) for passage in payload.passages]
    output = model.compute_score(
        pairs,
        batch_size=2,
        max_query_length=256,
        max_passage_length=2048,
        weights_for_different_modes=[0.0, 0.0, 1.0],
    )['colbert']
    if isinstance(output, (int, float)):
        output = [output]
    return {'scores': [float(score) for score in output]}

ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()
server = uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=8003, log_level='warning'))
threading.Thread(target=server.run, daemon=True).start()
public_url = ngrok.connect(8003, 'http').public_url
print('RAG_RERANKER_BASE_URL=' + public_url)